In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# مدخلات ومخرجات Estimator

<Accordion>
<AccordionItem title="إصدارات الحزم">

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
تُقدِّم هذه الصفحة نظرة عامة على مدخلات ومخرجات Qiskit Runtime Estimator primitive، الذي ينفِّذ أحمال العمل على موارد الحوسبة الخاصة بـ IBM Quantum&reg;. يتيح لك Estimator تعريف أحمال عمل متجهة بكفاءة باستخدام هيكل بيانات يُسمى [**كتلة Primitive الموحدة (PUB)**](/guides/primitive-input-output#pubs). تُستخدَم كمدخلات لطريقة [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) في Estimator primitive، التي تنفِّذ حمل العمل المُحدَّد كمهمة. ثم بعد اكتمال المهمة، تُعاد النتائج بتنسيق يعتمد على كل من PUBs المُستخدَمة وخيارات الـ runtime المُحدَّدة من الـ primitive.
## المدخلات
كل PUB بهذا التنسيق:

(`<circuit واحد>`، `<مراقب واحد أو أكثر>`، `<قيم معاملات واحدة أو أكثر اختيارية>`، `<دقة اختيارية>`),

يمكن أن تكون `parameter values` الاختيارية قائمة أو معاملاً واحدًا. تُجمَع عناصر المراقبات وقيم المعاملات باتباع قواعد بث NumPy كما هو موضح في موضوع [مدخلات ومخرجات الـ Primitive](primitive-input-output#broadcasting-rules)، ويُعاد تقدير قيمة توقع واحدة لكل عنصر من الشكل المُبثوث.

> **Note:** إذا كانت المدخلات تحتوي على قياسات، فإنها تُتجاهَل.

بالنسبة لـ Estimator primitive، يمكن أن يحتوي PUB على أربع قيم كحد أقصى:
- `QuantumCircuit` واحد، والذي قد يحتوي على كائن [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter) واحد أو أكثر
- قائمة من مراقب واحد أو أكثر، تُحدِّد قيم التوقع المُراد تقديرها، مُرتَّبة في مصفوفة (مثلاً مراقب واحد مُمثَّل كمصفوفة 0-d، وقائمة مراقبات كمصفوفة 1-d، وما إلى ذلك). يمكن أن تكون البيانات بأي تنسيق من تنسيقات `ObservablesArrayLike` كـ `Pauli` أو `SparsePauliOp` أو `PauliList` أو `str`.
   > **Note:** - المراقبات المتبادلة **في نفس PUB** تُجمَّع معًا باستخدام [هذه الطريقة](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting).
>    - المراقبات المتبادلة في PUBs مختلفة، حتى لو كانت لها نفس الـ Circuit، لا تُقدَّر باستخدام نفس القياس. كل PUB يُمثِّل أساسًا مختلفًا للقياس، وبالتالي تكون قياسات منفصلة مطلوبة لكل PUB.
>    - لضمان تقدير المراقبات المتبادلة باستخدام نفس القياس، جمِّعها داخل نفس PUB.
- مجموعة من قيم المعاملات لربط الـ Circuit بها. يمكن تحديد هذا كائن واحد شبيه بالمصفوفة حيث الفهرس الأخير هو على كائنات `Parameter` في الـ Circuit أو حذفه (أو بشكل مكافئ، ضبطه على `None`) إذا لم يكن في الـ Circuit كائنات `Parameter`.
- (اختياريًا) دقة مستهدفة لقيم التوقع المُراد تقديرها
---

الكود التالي يُوضِّح مثالاً على مجموعة من المدخلات المتجهة لـ `Estimator` primitive وينفِّذها على Backend من IBM&reg; كائن `RuntimeJobV2` واحد.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

## المخرجات
بعد إرسال PUB واحد أو أكثر إلى وحدة QPU للتنفيذ واكتمال مهمة بنجاح، تُعاد البيانات ككائن حاوية [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) يُمكن الوصول إليه باستدعاء طريقة `RuntimeJobV2.result()`.

يحتوي `PrimitiveResult` على قائمة قابلة للتكرار من كائنات [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) تحتوي على نتائج التنفيذ لكل PUB.

يتوافق كل عنصر في هذه القائمة مع كل PUB مُرسَل إلى طريقة `run()` في الـ primitive (مثلاً مهمة مُرسَلة بـ 20 PUB ستُعيد كائن `PrimitiveResult` يحتوي على قائمة من 20 كائن `PubResult`، واحد لكل PUB).

يحتوي كل `PubResult` في Estimator primitive على الأقل على مصفوفة من قيم التوقع (`PubResult.data.evs`) والانحرافات المعيارية المرتبطة بها (إما `PubResult.data.stds` أو `PubResult.data.ensemble_standard_error` بحسب `resilience_level` المُستخدَم)، لكن قد يحتوي على مزيد من البيانات بحسب خيارات تخفيف الأخطاء المُحدَّدة.

يمتلك كل كائن `PubResult` خاصيتَي `data` و`metadata`.
    - خاصية `data` هي [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) مخصصة تحتوي على قيم القياس الفعلية والانحرافات المعيارية وما إلى ذلك.
    - تمتلك `DataBin` خصائص متنوعة بحسب شكل أو هيكل PUB المرتبط به وخيارات تخفيف الأخطاء المُحدَّدة بواسطة الـ primitive المُستخدَم لإرسال المهمة (مثلاً [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) أو [PEC](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)).
    - تحتوي خاصية `metadata` على معلومات حول خيارات الـ runtime وتخفيف الأخطاء المُستخدَمة (موضَّحة لاحقًا في قسم [بيانات وصفية للنتيجة](#result-metadata) في هذه الصفحة).

فيما يلي مخطط مرئي لهيكل بيانات `PrimitiveResult` لمخرج Estimator:

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```

ببساطة، مهمة واحدة تُعيد كائن `PrimitiveResult` ويحتوي على قائمة من كائن `PubResult` واحد أو أكثر. تُخزِّن كائنات `PubResult` هذه بيانات القياس لكل PUB مُرسَل إلى المهمة.

مقتطف الكود التالي يصف تنسيق `PrimitiveResult` (وما يرتبط به من `PubResult`) للمهمة المُنشأة أعلاه.

In [2]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
with np.printoptions(threshold=200):
    print(
        f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
    )

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

#### How the Estimator primitive calculates error

In addition to the estimate of the mean of the observables passed in the input PUBs (the `evs` field of the `DataBin`), Estimator also attempts to deliver an estimate of the error associated with those expectation values. All Estimator queries will populate the `stds` field with a quantity like the standard error of the mean for each expectation value, but some error mitigation options produce additional information, such as `ensemble_standard_error`.

Consider a single observable $\mathcal{O}$. In the absence of [ZNE](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), you can think of each shot of the Estimator execution as providing a point estimate of the expectation value $\langle \mathcal{O} \rangle$. If the pointwise estimates are in a vector `Os`, then the value returned in `ensemble_standard_error` is equivalent to the following (in which $\sigma_{\mathcal{O}}$ is the [standard deviation of the expectation value](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) estimate and $N_{shots}$ is the number of shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

which treats all shots as part of a single ensemble. If you requested gate [twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), you can sort the pointwise estimates of $\langle \mathcal{O} \rangle$ into sets that share a common twirl. Call these sets of estimates `O_twirls`, and there are `num_randomizations` (number of twirls) of them. Then `stds` is the standard error of the mean of `O_twirls`, as in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

where $\sigma_{\mathcal{O}}$ is the standard deviation of `O_twirls` and $N_{twirls}$ is the number of twirls. When you do not enable twirling, `stds` and `ensemble_standard_error` are equal.

If you enable ZNE, then the `stds` described above become weights in a non-linear regression to an extrapolator model. What finally gets returned in the `stds` in this case is the uncertainty of the fit model evaluated at a noise factor of zero. When there is a poor fit, or large uncertainty in the fit, the reported `stds` can become very large. When ZNE is enabled, `pub_result.data.evs_noise_factors` and `pub_result.data.stds_noise_factors` are also populated, so that you can do your own extrapolation.

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [3]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'dynamical_decoupling' : {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'},
'twirling' : {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'},
'resilience' : {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False},
'version' : 2,

The metadata of the PubResult result is:
'shots' : 4096,
'target_precision' : 0.015625,
'circuit_metadata' : {},
'resilience' : {},
'num_randomizations' : 32,
